# Phase 1 — ASP Landscape: アリエールジェル vs アタック抗菌EX

**Purpose:** Map the complete pricing landscape per size for both brands.

| Step | Description |
|------|-------------|
| 1-1 | Monthly ASP trend per size — アリエールジェル (all sizes) |
| 1-2 | Monthly ASP trend per size — アタック抗菌EX (all sizes) |
| 1-3 | Side-by-side ASP comparison table: pre-renewal vs. post-renewal |
| 1-4 | Per-dose ASP gap (P&G capacity data + アタック ASP proxy) |

**ASP = `pos_sales_amt / pos_unit_sales_qty`** (no shelf_price_amt, no promo flag)  
**Created:** 2026-02-19

---
## 0. Imports & Connection

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
import databricks.sql as sql
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font: MS Gothic
✅ Credentials loaded


---
## 1. Parameters

In [2]:
ARIEL_GEL   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX   = 'ｱﾀｯｸ抗菌EX'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

ANALYSIS_START = '2025-01-01'
ANALYSIS_END   = '2026-01-31'

# ── Renewal breakpoint — update after Phase 0 confirmation ─────────────
RENEWAL_MONTH = '2025-05-01'  # Adjust based on Phase 0 findings

RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# ── Size role mapping (update after Phase 0) ──────────────────────────
# Maps each discovered size to its strategic role
SIZE_ROLE = {
    '本体通常':          'Trial',
    '詰替超特大':        'Trial',        # Competitive entry point
    '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ':  'Repeat',
    '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ': 'Repeat',
    # Add other sizes discovered in Phase 0 here
}

print(f'📋 Renewal breakpoint set to: {RENEWAL_MONTH}')
print(f'📋 Size roles configured: {len(SIZE_ROLE)} sizes')

📋 Renewal breakpoint set to: 2025-05-01
📋 Size roles configured: 4 sizes


---
## 2. Step 1-1 & 1-2: Monthly ASP Trend per Size (Both Brands)

In [3]:
# ── Fetch monthly ASP data for all sizes, both brands ─────────────────
asp_trend_query = f"""
SELECT
    DATE_TRUNC('month', CAST(idpos.sales_period_group_end_date_part AS DATE)) AS month,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    COUNT(DISTINCT idpos.shopper_key)   AS shoppers,
    SUM(idpos.pos_unit_sales_qty)       AS total_units,
    SUM(idpos.pos_sales_amt)            AS total_sales,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Fetching monthly ASP trends (all sizes, both brands)...', flush=True)
df_asp = execute_query(asp_trend_query)
df_asp['month'] = pd.to_datetime(df_asp['month'])
for col in ['shoppers', 'total_units', 'total_sales', 'weighted_asp']:
    df_asp[col] = pd.to_numeric(df_asp[col])

print(f'✅ {len(df_asp)} rows fetched')
print(f'   Ariel sizes: {df_asp[df_asp["sub_brand"]==ARIEL_GEL]["size_code"].nunique()}')
print(f'   Attack sizes: {df_asp[df_asp["sub_brand"]==ATTACK_EX]["size_code"].nunique()}')

⏳ Fetching monthly ASP trends (all sizes, both brands)...


HTTP request failed after retries: HTTPSConnectionPool(host='https', port=443): Max retries exceeded with url: //adb-2258763851730787.7.azuredatabricks.net/api/2.0/connector-service/feature-flags/PYTHON/4.2.3 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000016C38316320>: Failed to resolve 'https' ([Errno 11001] getaddrinfo failed)"))


✅ 182 rows fetched
   Ariel sizes: 8
   Attack sizes: 7


In [5]:
# ── Ariel Gel: ASP trend per size ─────────────────────────────────────
ariel_data = df_asp[df_asp['sub_brand'] == ARIEL_GEL].copy()
ariel_sizes = sorted(ariel_data['size_code'].unique())

n_sizes = len(ariel_sizes)
n_cols = 2
n_rows = (n_sizes + 1) // 2

fig = make_subplots(rows=n_rows, cols=n_cols,
                    subplot_titles=[f'アリエールジェル {s}' for s in ariel_sizes],
                    vertical_spacing=0.08, horizontal_spacing=0.08)

renewal_date = pd.Timestamp(RENEWAL_MONTH)

# Ensure tz-consistency
if ariel_data['month'].dt.tz is not None:
    renewal_date = renewal_date.tz_localize(ariel_data['month'].dt.tz)

for idx, size in enumerate(ariel_sizes):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    subset = ariel_data[ariel_data['size_code'] == size].sort_values('month')

    # Color code: blue pre-renewal, red post-renewal
    pre = subset[subset['month'] < renewal_date]
    post = subset[subset['month'] >= renewal_date]

    if len(pre) > 0:
        fig.add_trace(go.Scatter(x=pre['month'], y=pre['weighted_asp'],
                                 mode='lines+markers', name=f'{size} (Pre)',
                                 line=dict(color='#6495ED'), showlegend=(idx==0)),
                      row=row, col=col)
    if len(post) > 0:
        fig.add_trace(go.Scatter(x=post['month'], y=post['weighted_asp'],
                                 mode='lines+markers', name=f'{size} (Post)',
                                 line=dict(color='#FF6347'), showlegend=(idx==0)),
                      row=row, col=col)

    fig.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray',
                  opacity=0.5, row=row, col=col)

fig.update_layout(height=300*n_rows, title_text='アリエールジェル: Monthly ASP by Size (Pre vs Post Renewal)',
                  template='plotly_white')
fig.update_yaxes(title_text='ASP (JPY)')
fig.show()
# ── Summary table ─────────────────────────────────────────────────────
from IPython.display import display
_ariel_asp_tbl = ariel_data.groupby(['size_code', 'month']).agg(
    weighted_asp=('weighted_asp', 'first')
).reset_index()
_ariel_asp_tbl['month_str'] = _ariel_asp_tbl['month'].dt.strftime('%Y-%m')
_ariel_asp_tbl = _ariel_asp_tbl.pivot_table(index='size_code', columns='month_str', values='weighted_asp', aggfunc='first').round(0)
_ariel_asp_tbl.columns.name = 'Month'
_ariel_asp_tbl.index.name = 'Size'
print('アリエールジェル Monthly Weighted ASP (JPY):')
display(_ariel_asp_tbl)

In [6]:
# ── Attack 抗菌EX: ASP trend per size ────────────────────────────────
attack_data = df_asp[df_asp['sub_brand'] == ATTACK_EX].copy()
attack_sizes_list = sorted(attack_data['size_code'].unique())

if len(attack_sizes_list) > 0:
    n_s = len(attack_sizes_list)
    n_c = 2
    n_r = (n_s + 1) // 2

    fig2 = make_subplots(rows=n_r, cols=n_c,
                         subplot_titles=[f'アタック抗菌EX {s}' for s in attack_sizes_list],
                         vertical_spacing=0.08, horizontal_spacing=0.08)

    for idx, size in enumerate(attack_sizes_list):
        row = idx // n_c + 1
        col = idx % n_c + 1
        subset = attack_data[attack_data['size_code'] == size].sort_values('month')
        fig2.add_trace(go.Scatter(x=subset['month'], y=subset['weighted_asp'],
                                  mode='lines+markers', name=size,
                                  line=dict(color=px.colors.qualitative.Set2[idx % 8])),
                       row=row, col=col)

    fig2.update_layout(height=300*n_r, title_text='アタック抗菌EX: Monthly ASP by Size',
                       template='plotly_white')
    fig2.update_yaxes(title_text='ASP (JPY)')
    fig2.show()
else:
    print('⚠️ No Attack data found')
    # ── Summary table ─────────────────────────────────────────────────────
    from IPython.display import display
    _attack_asp_tbl = attack_data.groupby(['size_code', 'month']).agg(
        weighted_asp=('weighted_asp', 'first')
    ).reset_index()
    _attack_asp_tbl['month_str'] = _attack_asp_tbl['month'].dt.strftime('%Y-%m')
    _attack_asp_tbl = _attack_asp_tbl.pivot_table(index='size_code', columns='month_str', values='weighted_asp', aggfunc='first').round(0)
    _attack_asp_tbl.columns.name = 'Month'
    _attack_asp_tbl.index.name = 'Size'
    print('アタック抗菌EX Monthly Weighted ASP (JPY):')
    display(_attack_asp_tbl)

---
## 3. Step 1-3: Pre vs Post Renewal ASP Comparison Table

In [7]:
# ── Build comparison table ────────────────────────────────────────────
df_asp['period'] = df_asp['month'].apply(lambda x: 'Pre-Renewal' if x < renewal_date else 'Post-Renewal')

comparison = df_asp.groupby(['sub_brand', 'size_code', 'period']).agg(
    total_sales=('total_sales', 'sum'),
    total_units=('total_units', 'sum'),
    avg_shoppers_per_month=('shoppers', 'mean'),
).reset_index()

comparison['weighted_asp'] = comparison['total_sales'] / comparison['total_units']

# Pivot for side-by-side view
pivot = comparison.pivot_table(
    index=['sub_brand', 'size_code'],
    columns='period',
    values=['weighted_asp', 'avg_shoppers_per_month', 'total_units'],
    aggfunc='first'
).round(1)

# Calculate ASP change
flat = comparison.pivot_table(
    index=['sub_brand', 'size_code'],
    columns='period',
    values='weighted_asp',
    aggfunc='first'
).reset_index()

if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
    flat['asp_change_%'] = ((flat['Post-Renewal'] - flat['Pre-Renewal']) / flat['Pre-Renewal'] * 100).round(1)
    flat['asp_change_jpy'] = (flat['Post-Renewal'] - flat['Pre-Renewal']).round(0)

print('=' * 80)
print('ASP Comparison: Pre-Renewal vs Post-Renewal')
print(f'Pre : {ANALYSIS_START} → {RENEWAL_MONTH}')
print(f'Post: {RENEWAL_MONTH} → {ANALYSIS_END}')
print('=' * 80)
print()

# ── Ariel summary ──────────────────────────────────────────────────────
print('▶ アリエールジェル')
ariel_comp = flat[flat['sub_brand'] == ARIEL_GEL].copy()
print(ariel_comp.to_string(index=False))

print()
print('▶ アタック抗菌EX')
attack_comp = flat[flat['sub_brand'] == ATTACK_EX].copy()
print(attack_comp.to_string(index=False))

ASP Comparison: Pre-Renewal vs Post-Renewal
Pre : 2025-01-01 → 2025-05-01
Post: 2025-05-01 → 2026-01-31

▶ アリエールジェル
sub_brand     size_code  Post-Renewal  Pre-Renewal  asp_change_%  asp_change_jpy
ｱﾘｴｰﾙｼﾞｪﾙ          本体通常         279.1        216.5          28.9            63.0
ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大         330.8        341.4          -3.1           -11.0
ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         868.6        823.5           5.5            45.0
ｱﾘｴｰﾙｼﾞｪﾙ     詰替超ｼﾞｬﾝﾎﾞ         748.8        642.6          16.5           106.0
ｱﾘｴｰﾙｼﾞｪﾙ          詰替通常         217.5        145.4          49.6            72.0
ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         761.2        614.4          23.9           147.0
ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         942.8      1,024.2          -7.9           -81.0
ｱﾘｴｰﾙｼﾞｪﾙ           ｿﾉﾀ       1,931.0      2,694.5         -28.3          -763.0

▶ アタック抗菌EX
sub_brand     size_code  Post-Renewal  Pre-Renewal  asp_change_%  asp_change_jpy
 ｱﾀｯｸ抗菌EX          本体通常         269.1        345.2         -22

In [8]:
# ── Visualize ASP comparison as grouped bar chart ─────────────────────
if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
    plot_data = flat.copy()
    plot_data['label'] = plot_data['sub_brand'] + ' | ' + plot_data['size_code']

    fig3 = go.Figure()
    fig3.add_trace(go.Bar(name='Pre-Renewal ASP', x=plot_data['label'],
                          y=plot_data['Pre-Renewal'], marker_color='#6495ED'))
    fig3.add_trace(go.Bar(name='Post-Renewal ASP', x=plot_data['label'],
                          y=plot_data['Post-Renewal'], marker_color='#FF6347'))

    fig3.update_layout(barmode='group', title='ASP Before vs After Renewal — All Sizes',
                       yaxis_title='Weighted ASP (JPY)', template='plotly_white',
                       height=500)
    fig3.show()
    # ── Summary table ─────────────────────────────────────────────────────
    from IPython.display import display
    print('ASP Pre vs Post Renewal Comparison:')
    display(flat.round(1))
else:
    print('⚠️ Need both pre and post renewal data for comparison chart')

---
## 4. Step 1-4: Per-Dose ASP Gap (Ariel Capacity vs Attack Proxy)

In [9]:
# ── Fetch Ariel Gel capacity data from prod_dim (P&G only) ────────────
capacity_query = f"""
SELECT DISTINCT
    prod.jp_segment_4_name        AS size_code,
    prod.jp_prod_alter_lang_name  AS product_name,
    prod.jp_size_name             AS size_name,
    prod.jp_pack_size_name        AS pack_size_name
FROM id_pos_ai_1.prod_dim_ext_vw prod
WHERE prod.jp_sub_brand_alter_lang_name = '{ARIEL_GEL}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
ORDER BY 1, 2
"""

print('⏳ Fetching capacity data from product dimension...', flush=True)
df_cap = execute_query(capacity_query)
print(f'✅ {len(df_cap)} products found')
print()
print(df_cap.to_string(index=False))

⏳ Fetching capacity data from product dimension...


HTTP request error: 'NoneType' object has no attribute 'request'


✅ 430 products found

    size_code                                              product_name      size_name pack_size_name
         本体通常                                   ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 1.1kg             本体           None
         本体通常                                  ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 本体 1kg             本体           None
         本体通常                        ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 消臭成分ｱｯﾌﾟ 本体 850g             本体           None
         本体通常                              ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 部屋干ｼ用 900g             本体           None
         本体通常                 ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ 部屋干ｼ用 ｵﾘﾝﾋﾟｯｸﾛｺﾞ付ｷ 900g             本体           None
         本体通常                                ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ ｸｰﾙ 850g             本体           None
         本体通常                           ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ ｻｲｴﾝｽﾌﾟﾗｽ 1kg             本体           None
         本体通常                       ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ ｻｲｴﾝｽﾌﾟﾗｽ 本体 910g             本体           None
         本体通常                   ｱﾘｴｰﾙ ｲｵﾝﾊﾟﾜｰｼﾞｪﾙ ｻｲｴﾝｽﾌﾟﾗｽ 

In [10]:
# ── Build per-dose ASP table ──────────────────────────────────────────
# Manual capacity mapping — update based on Phase 0 product name inspection
# These are typical Ariel Gel capacities in grams/mL
ARIEL_CAPACITY_G = {
    '本体通常':          690,     # Typical 690g bottle
    '詰替超特大':        850,     # ~850g refill (competitive entry)
    '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ':  1260,    # ~1260g ultra jumbo refill
    '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ': 1520,    # ~1520g super ultra jumbo refill
    # Add more from Phase 0 product name inspection
}

print('⚠️  NOTE: Update ARIEL_CAPACITY_G based on actual product names from Phase 0.')
print('   Current values are typical estimates. Verify against jp_prod_alter_lang_name.')
print()

# Get post-renewal ASP per size for Ariel
ariel_post = df_asp[(df_asp['sub_brand'] == ARIEL_GEL) & (df_asp['month'] >= renewal_date)].copy()
ariel_asp_by_size = ariel_post.groupby('size_code').agg(
    total_sales=('total_sales', 'sum'),
    total_units=('total_units', 'sum')
).reset_index()
ariel_asp_by_size['weighted_asp'] = ariel_asp_by_size['total_sales'] / ariel_asp_by_size['total_units']

# Calculate per-gram ASP for Ariel
ariel_asp_by_size['capacity_g'] = ariel_asp_by_size['size_code'].map(ARIEL_CAPACITY_G)
ariel_asp_by_size['asp_per_gram'] = (ariel_asp_by_size['weighted_asp'] / ariel_asp_by_size['capacity_g']).round(2)
ariel_asp_by_size['sub_brand'] = ARIEL_GEL

# Attack: use ASP as proxy (no capacity data)
attack_post = df_asp[(df_asp['sub_brand'] == ATTACK_EX) & (df_asp['month'] >= renewal_date)].copy()
attack_asp_by_size = attack_post.groupby('size_code').agg(
    total_sales=('total_sales', 'sum'),
    total_units=('total_units', 'sum')
).reset_index()
attack_asp_by_size['weighted_asp'] = attack_asp_by_size['total_sales'] / attack_asp_by_size['total_units']
attack_asp_by_size['capacity_g'] = np.nan  # Not available for competitor
attack_asp_by_size['asp_per_gram'] = np.nan
attack_asp_by_size['sub_brand'] = ATTACK_EX

# Combine
dose_table = pd.concat([ariel_asp_by_size, attack_asp_by_size], ignore_index=True)

print('=' * 80)
print('Per-Dose ASP Comparison (Post-Renewal)')
print('  Ariel: ASP / capacity(g) | Attack: ASP only (no capacity data)')
print('=' * 80)
print(dose_table[['sub_brand', 'size_code', 'weighted_asp', 'capacity_g', 'asp_per_gram']].to_string(index=False))

⚠️  NOTE: Update ARIEL_CAPACITY_G based on actual product names from Phase 0.
   Current values are typical estimates. Verify against jp_prod_alter_lang_name.

Per-Dose ASP Comparison (Post-Renewal)
  Ariel: ASP / capacity(g) | Attack: ASP only (no capacity data)
sub_brand     size_code  weighted_asp  capacity_g  asp_per_gram
ｱﾘｴｰﾙｼﾞｪﾙ          本体通常         279.1       690.0           0.4
ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大         330.8       850.0           0.4
ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         868.6     1,520.0           0.6
ｱﾘｴｰﾙｼﾞｪﾙ     詰替超ｼﾞｬﾝﾎﾞ         748.8         NaN           NaN
ｱﾘｴｰﾙｼﾞｪﾙ          詰替通常         217.5         NaN           NaN
ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         761.2     1,260.0           0.6
ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         942.8         NaN           NaN
ｱﾘｴｰﾙｼﾞｪﾙ           ｿﾉﾀ       1,931.0         NaN           NaN
 ｱﾀｯｸ抗菌EX          本体通常         269.1         NaN           NaN
 ｱﾀｯｸ抗菌EX         詰替超特大         363.1         NaN           NaN
 ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ         

In [12]:
# ── Export Phase 1 results ────────────────────────────────────────────
output_file = 'phase1_asp_landscape.xlsx'

# Strip timezone for Excel compatibility
df_asp_export = df_asp.copy()
for col in df_asp_export.select_dtypes(include=['datetimetz']).columns:
    df_asp_export[col] = df_asp_export[col].dt.tz_localize(None)

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    df_asp_export.to_excel(writer, sheet_name='Monthly_ASP_Trend', index=False)
    flat.to_excel(writer, sheet_name='Pre_vs_Post_Comparison', index=False)
    dose_table.to_excel(writer, sheet_name='Per_Dose_ASP', index=False)

print(f'✅ Phase 1 results exported to {output_file}')

✅ Phase 1 results exported to phase1_asp_landscape.xlsx
